# M01-P02 — Structured Incident Analyzer

En P01 vimos que un LLM devuelve texto útil para una persona. Ahora queremos que una **aplicación** pueda utilizar la respuesta de forma fiable. Para ello pasaremos de texto libre a **JSON estructurado y validado** con Amazon Bedrock, JSON Schema y Pydantic.

> Si la sesión SSO ha caducado: `aws sso login --profile genai-course --use-device-code`


## 1. Preparar el cliente

Seguimos usando `boto3` y Converse directamente para mantener visible lo que ocurre.


In [ ]:
import json
import time
from typing import Literal

import boto3
import pandas as pd
from pydantic import BaseModel, ConfigDict, ValidationError

from enterprise_genai.config import get_settings

settings = get_settings()
session = boto3.Session(
    profile_name=settings.aws_profile or None,
    region_name=settings.aws_region,
)
runtime = session.client("bedrock-runtime", region_name=settings.aws_region)
identity = session.client("sts").get_caller_identity()
MODEL_ID = settings.bedrock_text_model_id

print("AWS identity:", identity["Arn"])
print("Region:", settings.aws_region)
print("Model:", MODEL_ID)


## 2. El problema del texto libre

Primero pedimos un análisis normal. La respuesta puede ser excelente para leer, pero nuestro programa no sabe dónde está exactamente la categoría, el número de afectados o las acciones.


In [ ]:
incident = (
    "Desde las 08:15, 73 usuarios de ACME no pueden autenticarse en la VPN. "
    "El servicio web corporativo funciona correctamente. "
    "El equipo de red informa de errores intermitentes en el backend de autenticación."
)

free_response = runtime.converse(
    modelId=MODEL_ID,
    system=[{"text": "Analiza incidencias empresariales sin inventar datos."}],
    messages=[
        {
            "role": "user",
            "content": [{"text": "Analiza esta incidencia:\n\n" + incident}],
        }
    ],
    inferenceConfig={"temperature": 0.0, "maxTokens": 300},
)

free_text = free_response["output"]["message"]["content"][0]["text"]
print(free_text)
print("\nTipo Python:", type(free_text).__name__)


### Pregunta

Si una aplicación necesita mostrar `affected_users` en una tarjeta o guardar `category` en una base de datos, ¿cómo extraería esos valores de un párrafo cuya redacción puede cambiar en cada ejecución?

Podríamos intentar analizar el texto manualmente, pero sería frágil. Es mejor pedir un **contrato de salida**.


## 3. Definir el contrato con Pydantic

Pydantic nos permite describir qué campos esperamos, qué tipos tienen y qué valores son válidos. `extra="forbid"` impide aceptar campos no definidos.


In [ ]:
class IncidentAnalysis(BaseModel):
    model_config = ConfigDict(extra="forbid")

    summary: str
    category: Literal[
        "authentication",
        "network",
        "application",
        "security",
        "other",
    ]
    affected_users: int | None
    affected_service: str | None
    observed_facts: list[str]
    hypotheses: list[str]
    recommended_actions: list[str]
    human_review_required: bool
    review_reason: str

schema = IncidentAnalysis.model_json_schema()
print(json.dumps(schema, indent=2, ensure_ascii=False))


### Qué debes observar

- `category` solo puede tomar uno de cinco valores.
- `affected_users` puede ser un entero o `null`.
- hechos, hipótesis y recomendaciones están separados.
- todos los campos son obligatorios, aunque algunos puedan contener `null`.
- `additionalProperties: false` evita campos inesperados.


## 4. Pedir structured output a Bedrock

Con `outputConfig.textFormat` enviamos el JSON Schema a Bedrock. La respuesta del modelo debe ajustarse a esa estructura. La primera vez que se utiliza un schema nuevo, Bedrock puede tardar más porque compila la gramática asociada.


In [ ]:
def run_structured_analysis(text: str):
    request = {
        "modelId": MODEL_ID,
        "system": [
            {
                "text": (
                    "Analiza incidencias empresariales usando únicamente la información disponible. "
                    "No inventes cifras, severidades, SLA ni causas raíz. "
                    "Si un dato no consta, usa null cuando el schema lo permita. "
                    "Separa hechos observados de hipótesis y marca revisión humana cuando "
                    "haya incertidumbre relevante."
                )
            }
        ],
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "text": (
                            "Convierte la siguiente incidencia en la estructura solicitada.\n\n"
                            + text
                        )
                    }
                ],
            }
        ],
        "inferenceConfig": {"temperature": 0.0, "maxTokens": 700},
        "outputConfig": {
            "textFormat": {
                "type": "json_schema",
                "structure": {
                    "jsonSchema": {
                        "schema": json.dumps(schema),
                        "name": "incident_analysis",
                        "description": "Structured analysis of an enterprise support incident",
                    }
                },
            }
        },
    }

    started = time.perf_counter()
    response = runtime.converse(**request)
    client_ms = round((time.perf_counter() - started) * 1000)
    raw_text = response["output"]["message"]["content"][0]["text"]
    validated = IncidentAnalysis.model_validate_json(raw_text)

    return {
        "raw_text": raw_text,
        "analysis": validated,
        "usage": response.get("usage", {}),
        "metrics": response.get("metrics", {}),
        "stop_reason": response.get("stopReason"),
        "client_ms": client_ms,
    }


In [ ]:
result = run_structured_analysis(incident)
analysis = result["analysis"]

print("--- JSON crudo devuelto por Bedrock ---")
print(result["raw_text"])

print("\n--- Objeto Python validado ---")
print(analysis.model_dump())

print("\nStop reason:", result["stop_reason"])
print("Usage:", result["usage"])
print("Bedrock metrics:", result["metrics"])
print("Client time (ms):", result["client_ms"])


## 5. ¿Qué gana ahora la aplicación?

Ya no tenemos que buscar datos dentro de un párrafo. Podemos acceder directamente a campos concretos.


In [ ]:
summary_table = pd.DataFrame(
    [
        ["category", analysis.category],
        ["affected_users", analysis.affected_users],
        ["affected_service", analysis.affected_service],
        ["human_review_required", analysis.human_review_required],
        ["review_reason", analysis.review_reason],
    ],
    columns=["campo", "valor"],
)
summary_table


## 6. Pydantic como segunda barrera

Bedrock restringe la salida con JSON Schema. Después Pydantic vuelve a validar localmente el resultado. Para ver qué aporta esa validación, creamos deliberadamente un objeto incorrecto sin llamar al modelo.


In [ ]:
invalid_example = {
    "summary": "Incidencia VPN",
    "category": "vpn",  # no está en el enum permitido
    "affected_users": "muchos",  # debería ser entero o null
    "affected_service": "VPN",
    "observed_facts": [],
    "hypotheses": [],
    "recommended_actions": [],
    "human_review_required": True,
    "review_reason": "Ejemplo deliberadamente inválido",
}

try:
    IncidentAnalysis.model_validate(invalid_example)
except ValidationError as exc:
    print(exc)


## 7. Segundo caso — cuando faltan datos

Una aplicación fiable no debería inventar el número de afectados. Probamos una incidencia donde ese dato no existe.


In [ ]:
incomplete_incident = (
    "Desde las 10:05 varios usuarios de BlueWave Logistics informan de lentitud al "
    "establecer la VPN. No se dispone todavía de un recuento de afectados ni hay "
    "confirmación de caída completa. El servicio web no presenta alertas. "
    "El equipo de operaciones está investigando."
)

result_missing = run_structured_analysis(incomplete_incident)
analysis_missing = result_missing["analysis"]

print(json.dumps(analysis_missing.model_dump(), indent=2, ensure_ascii=False))
print("\naffected_users:", analysis_missing.affected_users)
print("human_review_required:", analysis_missing.human_review_required)
print("stop_reason:", result_missing["stop_reason"])


### Observa

El resultado correcto para un dato ausente no es una estimación: es `null`. Esta distinción será esencial cuando conectemos el modelo con bases de datos, reglas o workflows.


## 8. Llevarlo a una interfaz

La página Streamlit **Analyze** reutiliza este patrón y convierte los campos estructurados en métricas, listas y paneles visuales.

Desde un terminal ejecuta:

```bash
streamlit run app/Home.py
```

Después abre **Analyze**, modifica la incidencia y observa tanto la vista amigable como los desplegables de JSON, schema y métricas.


# Conclusión

En P01 teníamos `LLM → texto`. Ahora tenemos:

`texto libre → Bedrock + JSON Schema → JSON válido → Pydantic → datos utilizables por la aplicación`

Esta es la primera versión funcional de **Analyze**. Más adelante ampliaremos la misma capacidad para documentos y entradas multimodales.
